# Automated Reasoning

Automated reasoning takes guardrails one step further. Where grounding asks "is this fact in the 
document?", automated reasoning asks "does this conclusion logically follow from the 
document?"

**Example:** A policy says "Collision deductible: 500 for standard plans, 250 for premium 
plans." A customer on a standard plan asks "What's my deductible?" If the model responds 
"$250", contextual grounding might score that highly — $250 IS in the document. Automated 
reasoning catches the error — $250 applies to premium plans, not this customer's plan.

**Grounding catches fabrication. Automated reasoning catches misapplication.**

In [28]:
# ============================================================
# Cell 2: Setup and Configuration
# ============================================================

import boto3
import json
import random
import string
import time

bedrock = boto3.client('bedrock', region_name='us-east-1')
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'
GUARDRAIL_ID = '18hmmqi7n3nu'

print(f"Model: {MODEL_ID}")
print(f"Guardrail: {GUARDRAIL_ID}")

Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0
Guardrail: 18hmmqi7n3nu


In [29]:
!pip install --upgrade boto3 botocore --break-system-packages

In [9]:
# ============================================================
# Cell 3: Create Automated Reasoning Policy
# ============================================================
# Two-step process via the API:
#   1. Create the policy with a skeleton definition
#   2. Start a build workflow that extracts formal logic from a source document
#
# The build workflow reads the document and translates business
# rules into formal logic automatically.

INSURANCE_RULES = """
INSURANCE POLICY RULES — AUTO COVERAGE

DEDUCTIBLES:
- Standard plan collision deductible: $500
- Premium plan collision deductible: $250
- Standard plan comprehensive deductible: $500
- Premium plan comprehensive deductible: $100

CLAIM PROCESSING TIMES:
- New claims require a minimum of 7 business days to process
- Emergency claims (towing, rental car) are processed within 2 business days
- Reopened claims require a minimum of 14 business days to process

COVERAGE LIMITS:
- Liability coverage maximum: $1,000,000 per occurrence
- Rental car reimbursement: maximum $40 per day for up to 30 days
- Total rental car reimbursement cannot exceed $1,200

ELIGIBILITY RULES:
- Drivers under 16 are not eligible for coverage
- Vehicles older than 25 years are not eligible for comprehensive coverage
- Commercial vehicles require a separate commercial policy
- Drivers with more than 3 at-fault accidents in the past 5 years are ineligible

EXCLUSIONS:
- Intentional damage is never covered
- Racing or competitive driving events are excluded
- Damage during commercial delivery use is excluded unless commercial policy is active
"""

# Step 1: Use the policy we already created
REASONING_POLICY_ARN = 'arn:aws:bedrock:us-east-1:360138725243:automated-reasoning-policy/gxyrpbwkjylk'
REASONING_POLICY_ID = 'gxyrpbwkjylk'
print(f"Policy ARN: {REASONING_POLICY_ARN}")
print(f"Policy ID: {REASONING_POLICY_ID}")

# Step 2: Start a build workflow to extract rules from the document
build_response = bedrock.start_automated_reasoning_policy_build_workflow(
    policyArn=REASONING_POLICY_ARN,
    buildWorkflowType='INGEST_CONTENT',
    sourceContent={
        'workflowContent': {
            'documents': [
                {
                    'document': INSURANCE_RULES.encode('utf-8'),
                    'documentContentType': 'txt',
                    'documentName': 'insurance-auto-coverage-rules',
                    'documentDescription': 'Auto insurance policy rules covering deductibles, processing times, coverage limits, eligibility, and exclusions'
                }
            ]
        }
    }
)

BUILD_WORKFLOW_ID = build_response['buildWorkflowId']
print(f"\nBuild workflow started!")
print(f"Workflow ID: {BUILD_WORKFLOW_ID}")
print("\nBedrock is extracting formal logic rules from the document...")
print("This may take a few minutes.")




Policy ARN: arn:aws:bedrock:us-east-1:360138725243:automated-reasoning-policy/gxyrpbwkjylk
Policy ID: gxyrpbwkjylk

Build workflow started!
Workflow ID: 51bb3e1e-1e33-4a2c-a8ca-a5c3682146be

Bedrock is extracting formal logic rules from the document...
This may take a few minutes.


In [12]:
# ============================================================
# Cell 4: Check Build Workflow Status
# ============================================================
# The build workflow extracts formal logic rules from the document.
# This takes a few minutes. Run this cell to check progress.

BUILD_WORKFLOW_ID = '51bb3e1e-1e33-4a2c-a8ca-a5c3682146be'

status_response = bedrock.get_automated_reasoning_policy_build_workflow(
    policyArn=REASONING_POLICY_ARN,
    buildWorkflowId=BUILD_WORKFLOW_ID
)

status = status_response.get('status', 'UNKNOWN')
print(f"Workflow ID: {BUILD_WORKFLOW_ID}")
print(f"Status: {status}")

if status == 'SUCCEEDED':
    print("\n✅ Build complete! Rules have been extracted.")
elif status == 'FAILED':
    print(f"\n❌ Build failed: {status_response.get('failureReason', 'unknown')}")
else:
    print("\n⏳ Still building... re-run this cell in a minute or two.")

print(f"\nFull response:")
print(json.dumps(status_response, indent=2, default=str))

Workflow ID: 51bb3e1e-1e33-4a2c-a8ca-a5c3682146be
Status: COMPLETED

⏳ Still building... re-run this cell in a minute or two.

Full response:
{
  "ResponseMetadata": {
    "RequestId": "99de5cba-1de8-4fd7-aec7-127d13a05166",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 10 Apr 2026 14:45:59 GMT",
      "content-type": "application/json",
      "content-length": "361",
      "connection": "keep-alive",
      "x-amzn-requestid": "99de5cba-1de8-4fd7-aec7-127d13a05166"
    },
    "RetryAttempts": 0
  },
  "policyArn": "arn:aws:bedrock:us-east-1:360138725243:automated-reasoning-policy/gxyrpbwkjylk",
  "buildWorkflowId": "51bb3e1e-1e33-4a2c-a8ca-a5c3682146be",
  "status": "COMPLETED",
  "buildWorkflowType": "INGEST_CONTENT",
  "createdAt": "2026-04-10 14:25:59.782000+00:00",
  "updatedAt": "2026-04-10 14:31:17.113000+00:00"
}


In [16]:
# ============================================================
# Cell 5: Retrieve Extracted Rules from Build Workflow
# ============================================================
# The build workflow produces several assets. Let's retrieve
# the policy definition (extracted rules) and the quality report.

BUILD_WORKFLOW_ID = '51bb3e1e-1e33-4a2c-a8ca-a5c3682146be'

# First, get the asset manifest to see what was produced
manifest = bedrock.get_automated_reasoning_policy_build_workflow_result_assets(
    policyArn=REASONING_POLICY_ARN,
    buildWorkflowId=BUILD_WORKFLOW_ID,
    assetType='ASSET_MANIFEST'
)

print("AVAILABLE ASSETS:")
for entry in manifest['buildWorkflowAssets']['assetManifest']['entries']:
    print(f"  {entry['assetType']}: {entry.get('assetName', 'N/A')} (ID: {entry.get('assetId', 'N/A')})")

# Now get the extracted policy definition (rules, variables, types)
policy_def = bedrock.get_automated_reasoning_policy_build_workflow_result_assets(
    policyArn=REASONING_POLICY_ARN,
    buildWorkflowId=BUILD_WORKFLOW_ID,
    assetType='POLICY_DEFINITION'
)

definition = policy_def['buildWorkflowAssets']['policyDefinition']

# Display variables
variables = definition.get('variables', [])
print(f"\nVARIABLES EXTRACTED: {len(variables)}")
print("-" * 60)
for v in variables:
    print(f"  {v['name']} ({v['type']}): {v.get('description', 'N/A')}")

# Display custom types
types = definition.get('types', [])
print(f"\nCUSTOM TYPES EXTRACTED: {len(types)}")
print("-" * 60)
for t in types:
    print(f"  {t['name']}: {t.get('description', 'N/A')}")
    for val in t.get('values', []):
        print(f"    - {val['value']}: {val.get('description', '')}")

# Display rules
rules = definition.get('rules', [])
print(f"\nRULES EXTRACTED: {len(rules)}")
print("-" * 60)
for r in rules:
    print(f"\n  Rule: {r['id']}")
    print(f"  Expression: {r['expression']}")
    if r.get('alternateExpression'):
        print(f"  Plain English: {r['alternateExpression']}")

# Also get the quality report
try:
    quality = bedrock.get_automated_reasoning_policy_build_workflow_result_assets(
        policyArn=REASONING_POLICY_ARN,
        buildWorkflowId=BUILD_WORKFLOW_ID,
        assetType='QUALITY_REPORT'
    )
    qr = quality['buildWorkflowAssets']['qualityReport']
    print(f"\nQUALITY REPORT:")
    print(f"  Types: {qr.get('typeCount', 0)}")
    print(f"  Variables: {qr.get('variableCount', 0)}")
    print(f"  Rules: {qr.get('ruleCount', 0)}")
    print(f"  Unused types: {qr.get('unusedTypes', [])}")
    print(f"  Unused variables: {qr.get('unusedVariables', [])}")
    print(f"  Conflicting rules: {qr.get('conflictingRules', [])}")
except Exception as e:
    print(f"\nCouldn't retrieve quality report: {e}")

AVAILABLE ASSETS:
  SOURCE_DOCUMENT: 21c19b43_source_document.json (ID: 21c19b43)
  BUILD_LOG: mutation_log.json (ID: N/A)
  POLICY_DEFINITION: policy.json (ID: N/A)
  QUALITY_REPORT: quality_report.json (ID: N/A)
  GENERATED_TEST_CASES: tests.json (ID: N/A)

VARIABLES EXTRACTED: 34
------------------------------------------------------------
  standardPlanCollisionDeductible (INT): The collision deductible amount in dollars for the standard insurance plan
  premiumPlanCollisionDeductible (INT): The collision deductible amount in dollars for the premium insurance plan
  standardPlanComprehensiveDeductible (INT): The comprehensive deductible amount in dollars for the standard insurance plan
  premiumPlanComprehensiveDeductible (INT): The comprehensive deductible amount in dollars for the premium insurance plan
  planType (PlanType): The type of insurance plan the customer has
  collisionDeductible (INT): The actual collision deductible amount in dollars that applies to the customer base

In [24]:
# ============================================================
# Cell 7: Publish Policy Version and Attach to Guardrail
# ============================================================
# The rules are on the policy (definition hash changed).
# Now we need to:
#   1. Publish a version using the definitionHash
#   2. Attach the policy to our guardrail
#   Note: Automated reasoning requires cross-region inference.

# Step 1: Publish a version
DEFINITION_HASH = '7d285993e97b866d8ffd8c56463ca59e9b9892d6062ec36d5c2eabf973c56f8984250246dc6bfdce4416ee4e14bbce79dc73262abdb3f91535c6c634645aec30'

try:
    version_response = bedrock.create_automated_reasoning_policy_version(
        policyArn=REASONING_POLICY_ARN,
        lastUpdatedDefinitionHash=DEFINITION_HASH
    )
    REASONING_VERSION = version_response.get('version', 'unknown')
    print(f"Published version: {REASONING_VERSION}")
except Exception as e:
    print(f"Version creation note: {e}")
    print("(May already exist from previous run)")

# Step 2: Attach to guardrail with cross-region inference
existing = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion='DRAFT')
content_policy = existing.get('contentPolicy', {})
topic_policy = existing.get('topicPolicy', {})
sensitive_policy = existing.get('sensitiveInformationPolicy', {})
word_policy = existing.get('wordPolicy', {})

# Cross-region guardrail profile for US regions
CROSS_REGION_PROFILE = 'us.guardrail.v1:0'

update_response = bedrock.update_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    name=existing['name'],
    blockedInputMessaging=existing['blockedInputMessaging'],
    blockedOutputsMessaging=existing['blockedOutputsMessaging'],
    contentPolicyConfig={'filtersConfig': content_policy.get('filters', [])},
    topicPolicyConfig={'topicsConfig': topic_policy.get('topics', [])},
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': sensitive_policy.get('piiEntities', []),
        'regexesConfig': sensitive_policy.get('regexes', [])
    },
    wordPolicyConfig={
        'wordsConfig': word_policy.get('words', []),
        'managedWordListsConfig': word_policy.get('managedWordLists', [])
    },
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {'type': 'GROUNDING', 'threshold': 0.7},
            {'type': 'RELEVANCE', 'threshold': 0.7}
        ]
    },
    automatedReasoningPolicyConfig={
        'policies': [REASONING_POLICY_ARN]
    },
    crossRegionConfig={
        'guardrailProfileIdentifier': CROSS_REGION_PROFILE
    }
)

print(f"\nGuardrail updated: {update_response['guardrailId']}")
print(f"Version: {update_response['version']}")

# Verify automated reasoning is attached
verify = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion='DRAFT')
ar_policy = verify.get('automatedReasoningPolicy', {})
print(f"\nAutomated Reasoning attached: {bool(ar_policy)}")
print(f"Policy details: {json.dumps(ar_policy, indent=2, default=str)}")

Published version: 5

Guardrail updated: 18hmmqi7n3nu
Version: DRAFT

Automated Reasoning attached: True
Policy details: {
  "policies": [
    "arn:aws:bedrock:us-east-1:360138725243:automated-reasoning-policy/gxyrpbwkjylk"
  ]
}


In [31]:
# ============================================================
# Cell 8: Test Automated Reasoning — Correct vs Incorrect
# ============================================================
# Test 1: Correct answer — Standard plan collision deductible is $500
# Test 2: Incorrect answer — Claims standard plan deductible is $250
#          ($250 is the PREMIUM plan deductible — classic misapplication)

def test_reasoning(query, response_text, label):
    """Test a response against the automated reasoning policy."""
    result = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion='DRAFT',
        source='OUTPUT',
        content=[
            {'text': {'text': INSURANCE_RULES, 'qualifiers': ['grounding_source']}},
            {'text': {'text': query, 'qualifiers': ['query']}},
            {'text': {'text': response_text, 'qualifiers': ['guard_content']}}
        ]
    )
    
    print(f"\n{'='*60}")
    print(f"TEST: {label}")
    print(f"QUERY: {query}")
    print(f"RESPONSE: {response_text}")
    print(f"ACTION: {result['action']}")
    
    for assessment in result.get('assessments', []):
        if 'automatedReasoningPolicy' in assessment:
            ar = assessment['automatedReasoningPolicy']
            print(f"\nAUTOMATED REASONING:")
            print(json.dumps(ar, indent=2, default=str))
        if 'contextualGroundingPolicy' in assessment:
            cg = assessment['contextualGroundingPolicy']
            print(f"\nCONTEXTUAL GROUNDING:")
            for f in cg.get('filters', []):
                print(f"  {f['type']}: score={f['score']}, action={f['action']}")
    
    return result

# Test 1: Correct — Standard plan = $500 collision deductible
test_reasoning(
    "I have a standard plan. What is my collision deductible?",
    "Your collision deductible on the standard plan is $500.",
    "CORRECT — Standard plan collision deductible"
)

# Test 2: Incorrect — Claims $250 but that's the PREMIUM deductible
test_reasoning(
    "I have a standard plan. What is my collision deductible?",
    "Your collision deductible is $250.",
    "INCORRECT — Wrong deductible for standard plan"
)

# Test 3: Incorrect — Claims new claims process in 3 days (minimum is 7)
test_reasoning(
    "How quickly can you process my new claim?",
    "We can process your new claim within 3 business days.",
    "INCORRECT — Under minimum processing time"
)


TEST: CORRECT — Standard plan collision deductible
QUERY: I have a standard plan. What is my collision deductible?
RESPONSE: Your collision deductible on the standard plan is $500.
ACTION: NONE

AUTOMATED REASONING:
{
  "findings": [
    {
      "valid": {
        "translation": {
          "premises": [
            {
              "logic": "(= planType STANDARD)",
              "naturalLanguage": "planType is equal to STANDARD"
            }
          ],
          "claims": [
            {
              "logic": "(= collisionDeductible\n   500)",
              "naturalLanguage": "collisionDeductible is equal to 500"
            }
          ],
          "untranslatedPremises": [],
          "untranslatedClaims": [],
          "confidence": 1.0
        },
        "claimsTrueScenario": {
          "statements": [
            {
              "logic": "(= liabilityCoverageMaximum\n   1000000)",
              "naturalLanguage": "liabilityCoverageMaximum is equal to 1000000"
            },


{'ResponseMetadata': {'RequestId': 'c7741383-53b4-411c-b95a-c55ed9e193e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 10 Apr 2026 18:11:27 GMT',
   'content-type': 'application/json',
   'content-length': '2295',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c7741383-53b4-411c-b95a-c55ed9e193e8'},
  'RetryAttempts': 0},
 'usage': {'topicPolicyUnits': 1,
  'contentPolicyUnits': 1,
  'wordPolicyUnits': 1,
  'sensitiveInformationPolicyUnits': 1,
  'sensitiveInformationPolicyFreeUnits': 1,
  'contextualGroundingPolicyUnits': 2,
  'contentPolicyImageUnits': 0,
  'automatedReasoningPolicyUnits': 1,
  'automatedReasoningPolicies': 1},
 'action': 'GUARDRAIL_INTERVENED',
 'actionReason': 'Guardrail blocked.',
 'outputs': [{'text': "I'm sorry, I can't provide that response. Let me help you with your insurance question in a different way."}],
 'assessments': [{'contextualGroundingPolicy': {'filters': [{'type': 'GROUNDING',
      'threshold': 0.7,
      'score': 0.02,
      

In [27]:
# ============================================================
# Cell 9: Automated Reasoning Test Suite
# ============================================================
# Test across all the business rules: deductibles, processing
# times, coverage limits, eligibility, and exclusions.

test_cases = [
    # DEDUCTIBLE TESTS
    ("CORRECT — Standard collision deductible",
     "I have a standard plan. What's my collision deductible?",
     "Your collision deductible is $500."),
    
    ("INCORRECT — Wrong deductible for standard plan",
     "I have a standard plan. What's my collision deductible?",
     "Your collision deductible is $250."),
    
    ("CORRECT — Premium comprehensive deductible",
     "I'm on the premium plan. What's my comprehensive deductible?",
     "Your comprehensive deductible on the premium plan is $100."),
    
    ("INCORRECT — Standard deductible given for premium plan",
     "I'm on the premium plan. What's my comprehensive deductible?",
     "Your comprehensive deductible is $500."),
    
    # PROCESSING TIME TESTS
    ("CORRECT — New claim 7 days",
     "How long will my new claim take to process?",
     "New claims require a minimum of 7 business days to process."),
    
    ("INCORRECT — New claim 3 days",
     "How long will my new claim take to process?",
     "We can process your new claim in 3 business days."),
    
    ("CORRECT — Emergency claim 2 days",
     "I need a rental car urgently. How fast can you process it?",
     "Emergency claims like rental car coverage are processed within 2 business days."),
    
    # COVERAGE LIMIT TESTS
    ("CORRECT — Rental car $40/day max 30 days",
     "What's my rental car coverage?",
     "Your rental car reimbursement is up to $40 per day for a maximum of 30 days."),
    
    ("INCORRECT — Rental car $60/day",
     "What's my rental car coverage?",
     "Your rental car reimbursement is $60 per day."),
    
    # ELIGIBILITY TESTS
    ("CORRECT — Under 16 not eligible",
     "My son is 15. Can he be on my policy?",
     "Drivers under 16 are not eligible for coverage."),
    
    ("INCORRECT — Under 16 eligible",
     "My son is 15. Can he be on my policy?",
     "Yes, your 15-year-old son can be added to your policy."),
    
    # EXCLUSION TESTS
    ("CORRECT — Intentional damage excluded",
     "Is intentional damage covered?",
     "No, intentional damage to your vehicle is never covered under the policy."),
    
    ("INCORRECT — Intentional damage covered",
     "Is intentional damage covered?",
     "Yes, all damage to your vehicle is covered regardless of cause."),
]

print("=" * 70)
print("AUTOMATED REASONING TEST SUITE")
print("=" * 70)

for label, query, response_text in test_cases:
    result = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion='DRAFT',
        source='OUTPUT',
        content=[
            {'text': {'text': INSURANCE_RULES, 'qualifiers': ['grounding_source']}},
            {'text': {'text': query, 'qualifiers': ['query']}},
            {'text': {'text': response_text, 'qualifiers': ['guard_content']}}
        ]
    )
    
    action = result['action']
    
    # Extract automated reasoning finding type
    ar_result = '—'
    ar_confidence = '—'
    ar_rule = '—'
    for assessment in result.get('assessments', []):
        if 'automatedReasoningPolicy' in assessment:
            findings = assessment['automatedReasoningPolicy'].get('findings', [])
            if findings:
                finding = findings[0]
                if 'valid' in finding:
                    ar_result = 'VALID'
                    ar_confidence = finding['valid']['translation'].get('confidence', '—')
                    rules = finding['valid'].get('supportingRules', [])
                    ar_rule = rules[0]['identifier'] if rules else '—'
                elif 'invalid' in finding:
                    ar_result = 'INVALID'
                    ar_confidence = finding['invalid']['translation'].get('confidence', '—')
                    rules = finding['invalid'].get('contradictingRules', [])
                    ar_rule = rules[0]['identifier'] if rules else '—'
                elif 'translationAmbiguous' in finding:
                    ar_result = 'AMBIGUOUS'
                elif 'satisfiable' in finding:
                    ar_result = 'SATISFIABLE'
    
    icon = '✅' if action == 'NONE' else '🛑'
    print(f"\n{icon} {label}")
    print(f"   Response:  {response_text[:65]}...")
    print(f"   Action:    {action}")
    print(f"   Reasoning: {ar_result} (confidence: {ar_confidence}, rule: {ar_rule})")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

AUTOMATED REASONING TEST SUITE

✅ CORRECT — Standard collision deductible
   Response:  Your collision deductible is $500....
   Action:    NONE
   Reasoning: VALID (confidence: 1.0, rule: IP7BV9301JIU)

🛑 INCORRECT — Wrong deductible for standard plan
   Response:  Your collision deductible is $250....
   Action:    GUARDRAIL_INTERVENED
   Reasoning: INVALID (confidence: 1.0, rule: IP7BV9301JIU)

✅ CORRECT — Premium comprehensive deductible
   Response:  Your comprehensive deductible on the premium plan is $100....
   Action:    NONE
   Reasoning: AMBIGUOUS (confidence: —, rule: —)

✅ INCORRECT — Standard deductible given for premium plan
   Response:  Your comprehensive deductible is $500....
   Action:    NONE
   Reasoning: INVALID (confidence: 1.0, rule: R6USH0BL8F27)

✅ CORRECT — New claim 7 days
   Response:  New claims require a minimum of 7 business days to process....
   Action:    NONE
   Reasoning: VALID (confidence: 1.0, rule: CHAPGM6AF8GA)

🛑 INCORRECT — New claim 3 days
 

## Day 8 Summary

**What we built:** An Automated Reasoning policy that translates insurance business rules 
into formal logic, then validates model responses against those rules using theorem proving — 
not pattern matching.

**Setup process (4 steps):**
1. **Create policy** — empty skeleton via `create_automated_reasoning_policy`
2. **Build workflow** — feed source document via `start_automated_reasoning_policy_build_workflow`, 
   Bedrock extracts variables, types, and formal logic rules automatically
3. **Apply rules** — update the policy with extracted definition
4. **Attach to guardrail** — add to existing guardrail with `crossRegionConfig` for cross-region inference

**What Bedrock extracted from a plain text document:**
- 34 variables (deductibles, processing times, eligibility flags, coverage limits)
- 5 custom enum types (PlanType, ClaimType, CoverageType, VehicleType, DamageType)
- 41 formal logic rules (if/then constraints, boundary conditions, compound calculations)
- 0 conflicting rules

**Key distinction — Grounding vs Automated Reasoning:**
- **Grounding** asks: "Is this fact in the source document?" (similarity check)
- **Automated Reasoning** asks: "Does this conclusion logically follow?" (theorem proving)
- Grounding catches fabrication. Automated reasoning catches misapplication.
- Automated reasoning returns VALID, INVALID, AMBIGUOUS, or SATISFIABLE — and cites 
  the specific rule that supports or contradicts the response.

**Test results (13 cases):**
- Correct responses validated with confidence 1.0 and supporting rule citations
- Incorrect deductibles, processing times, and exclusions caught with contradicting rules
- Some gaps: conversational phrasing ("your 15-year-old son") harder to translate than 
  explicit statements ("the driver is 15 years old")
- False positive on correct rental car response — contextual grounding too strict

**What's next (Day 9):** Red teaming — adversarial testing of all guardrail layers.